# Phase 3: passive Clifford actions for nonuniform CM records

This notebook applies the validated gate engine to six exact nonuniform CM candidates from the systole project. The output separates geometric automorphism order, the image on $K(L)$, and coverage of the full finite symplectic target.

In [1]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
candidates = (here, here.parent, here / 'passive-cliffords')
project = next(path for path in candidates if (path / 'src' / 'gkp_passive_cliffords').exists())
workspace = project.parent
sys.path.insert(0, str(project / 'src'))
sys.path.insert(0, str(workspace / 'src'))

from gkp_passive_cliffords import phase3_cm_action_table

## Exact nonuniform survey

The target column is $|\mathrm{Sp}(K(L))|$. Coverage one means that passive Gaussian transformations realize every symplectic action on the logical Pauli group, modulo Paulis and phases.

In [2]:
rows = phase3_cm_action_table()
columns = [
    'candidate_id', 'polarization_type', 'polarized_automorphism_order',
    'logical_image_order', 'action_kernel_order',
    'full_symplectic_target_order', 'target_coverage', 'ell_squared_exact',
]
print(' | '.join(columns))
print(' | '.join(['---'] * len(columns)))
for row in rows:
    print(' | '.join(str(row[column]) for column in columns))

candidate_id | polarization_type | polarized_automorphism_order | logical_image_order | action_kernel_order | full_symplectic_target_order | target_coverage | ell_squared_exact
--- | --- | --- | --- | --- | --- | --- | ---
g2_type_13_delta_24 | (1, 3) | 24 | 24 | 1 | 24 | 1 | 4/sqrt(24)
g2_type_15_reconstructed | (1, 5) | 24 | 24 | 1 | 120 | 1/5 | 2/sqrt(10)
g3_type_112_reconstructed | (1, 1, 2) | 12 | 3 | 4 | 6 | 1/2 | 2/sqrt(3)
g3_type_112_gaussian_bounded | (1, 1, 2) | 384 | 6 | 64 | 6 | 1 | 2/sqrt(4) = 1
g3_type_113_eisenstein_bounded | (1, 1, 3) | 1296 | 24 | 54 | 24 | 1 | 2/sqrt(3)
g3_type_122_gaussian_bounded | (1, 2, 2) | 384 | 48 | 8 | 720 | 1/15 | 2/sqrt(4) = 1


## Which CM records saturate the logical target?

In [3]:
for row in rows:
    status = 'FULL' if row['target_saturated'] else f"{row['target_coverage']} of target"
    print(f"{row['candidate_id']:36s} {status}")

g2_type_13_delta_24                  FULL
g2_type_15_reconstructed             1/5 of target
g3_type_112_reconstructed            1/2 of target
g3_type_112_gaussian_bounded         FULL
g3_type_113_eisenstein_bounded       FULL
g3_type_122_gaussian_bounded         1/15 of target


## First robustness--gate tradeoff

The two type-$(1,1,2)$ candidates provide a controlled comparison at fixed $D$. The reconstructed CM record improves $\ell^2$ from $1$ to $2/\sqrt3$, but its passive image decreases from the full order-six target to order three. Thus the more robust candidate is not the more gate-rich candidate.

In [4]:
type112 = [row for row in rows if row['polarization_type'] == (1, 1, 2)]
for row in type112:
    print(row['candidate_id'], 'ell^2 =', row['ell_squared_exact'],
          'image =', row['logical_image_order'], '/', row['full_symplectic_target_order'])
assert len(type112) == 2

g3_type_112_reconstructed ell^2 = 2/sqrt(3) image = 3 / 6
g3_type_112_gaussian_bounded ell^2 = 2/sqrt(4) = 1 image = 6 / 6


## Reproducibility checks

In [5]:
expected = {
    'g2_type_13_delta_24': (24, 24, 1, 24),
    'g2_type_15_reconstructed': (24, 24, 1, 120),
    'g3_type_112_reconstructed': (12, 3, 4, 6),
    'g3_type_112_gaussian_bounded': (384, 6, 64, 6),
    'g3_type_113_eisenstein_bounded': (1296, 24, 54, 24),
    'g3_type_122_gaussian_bounded': (384, 48, 8, 720),
}
for row in rows:
    observed = (row['polarized_automorphism_order'], row['logical_image_order'],
                row['action_kernel_order'], row['full_symplectic_target_order'])
    assert observed == expected[row['candidate_id']], (row['candidate_id'], observed)
    assert row['pairing_verified']
print('All Phase 3 checks passed.')

All Phase 3 checks passed.
